In [298]:
import pandas as pd
import numpy as np
from pathlib import Path

In [299]:
# data load
base_data = pd.read_csv("./fsae_data/2025/raw-data.csv")

cost_data = pd.read_csv("./fsae_data/2025/fsae_cost_results.csv")
# cost_data = cost_data.drop(columns=["Place", "Car Num", "Penalty", "Score", "Penalty"])
cost_data = cost_data[["Team", "Adjusted Cost"]]
merged_data = pd.merge(base_data, cost_data, on = 'Team', how = "outer")

efficiency_data = pd.read_csv("./fsae_data/2025/fsae_efficiency_results.csv")
#efficiency_data = efficiency_data.drop(columns = ["Place", "Car Num", "Fuel Efficiency Score"])
efficiency_data = efficiency_data[["Team", "Average Adjusted Laptime", "Fuel Efficiency Factor"]]
merged_data = pd.merge(merged_data, efficiency_data, on= 'Team', how = 'outer')

presentation_data = pd.read_csv("./fsae_data/2025/fsae_presentation_results.csv")
presentation_data = presentation_data.drop(columns = ["Place", "Car Num", "Score", "Penalty", "Raw Score", "Status"])
merged_data = pd.merge(merged_data, presentation_data, on= 'Team', how = 'outer')

accel_data = pd.read_csv("./fsae_data/2025/fsae_accel_results.csv", index_col=False)
accel_data = accel_data[["Team", "Final Best Time"]]
accel_data = accel_data.rename(columns = {"Final Best Time": "Best Acceleration Time"})
#accel_data = accel_data.drop(columns = ["Place", "Car Num", "Run 1 Time", "Run 1 Adj Time", "Run 2 Time", "Run 2 Adj Time", "Run 3 Time", "Run 3 Adj Time", "Run 4 Time", "Run 4 Adj Time"])
merged_data = pd.merge(merged_data, accel_data, on= 'Team', how = 'outer')

autocross_data = pd.read_csv("./fsae_data/2025/fsae_autocross_results.csv", index_col=False)
autocross_data = autocross_data[["Team", "Best Time"]]
autocross_data = autocross_data.rename(columns = {"Best Time": "Best Autocross Time"})
#autocross_data = autocross_data.drop(columns = ["Place", "Car Num", "Penalty","R1 Time","R1 Cones","R1 Off Course","R1 Adj Time","R2 Time","R2 Cones","R2 Off Course","R2 Adj Time","R3 Time","R3 Cones","R3 Off Course","R3 Adj Time","R4 Time","R4 Cones","R4 Off Course","R4 Adj Time"])
merged_data = pd.merge(merged_data, autocross_data, on= 'Team', how = 'outer')

endurance_data = pd.read_csv("./fsae_data/2025/fsae_endurance_results.csv", index_col=False)
#endurance_data = endurance_data.drop(columns = ["Place", "Car Num", "Cones", "Endurance Score"])
endurance_data = endurance_data[["Team", "Time", "Other Penalty", "Adjusted Time", "Time Score", "Laps", "Laps Score"]]
endurance_data = endurance_data.rename(columns = {"Time": "Endurance Time", "Other Penalty": "Endurance Penalty", "Adjusted Time": "Adjusted Endurance Time", "Time Score": "Endurance Time Score", "Laps": "Endurance Laps", "Laps Score": "Endurance Laps Score"})
merged_data = pd.merge(merged_data, endurance_data, on= 'Team', how = 'outer')

skidpad_data = pd.read_csv("./fsae_data/2025/fsae_skidpad_results.csv", index_col=False)
skidpad_data = skidpad_data[["Team", "Best Time"]]
skidpad_data = skidpad_data.rename(columns = {"Best Time": "Best Skidpad Time"})
merged_data = pd.merge(merged_data, skidpad_data, on= 'Team', how = 'outer')


team_data = pd.read_csv("./fsae_data/2025/fsae_team_info.csv", index_col=False)
team_data = team_data[["Team", "Engine Cylinders", "Engine Displacement (cc)", "Weight (kg)"]]
merged_data = pd.merge(merged_data, team_data, on= 'Team', how = 'outer')

merged_data["Team"] = merged_data["Team"] + "_2025"
#print(merged_data)

In [300]:
# Clean place data
placement_new = pd.DataFrame.copy(merged_data)
placement_new = placement_new.dropna(subset=["Place"])
unranked = merged_data[merged_data["Place"] == "Unranked"]
placement_new = placement_new[placement_new.Place != "Unranked"]

tied_cases = placement_new["Place"].str.contains('T')
#tied_cases = tied_cases.fillna(False)
tied = placement_new[tied_cases]
tied["Place"] = tied["Place"].str[:-2]
placement_new = placement_new[~tied_cases]
placement_new = pd.concat([placement_new, tied], ignore_index = True)

placement_new["Place"] = placement_new["Place"].astype(int)
placement_new = placement_new.reset_index(drop = True)
adjustment = pd.DataFrame.copy(placement_new)
for i in range(len(placement_new)):
    if placement_new.iloc[i]["Place"] == 1:
        adjustment.at[i,"Place"] = "1"
    elif placement_new.iloc[i]["Place"] <= 5:
        adjustment.at[i,"Place"] = "<=5"
    elif placement_new.iloc[i]["Place"] <= 10:
        adjustment.at[i,"Place"] = "<=10"
    elif placement_new.iloc[i]["Place"] <= 25:
        adjustment.at[i,"Place"] = "<=25"
    elif placement_new.iloc[i]["Place"] <= 50:
        adjustment.at[i,"Place"] = "<=50"
    elif placement_new.iloc[i]["Place"] <= 75:
        adjustment.at[i,"Place"] = "<=75"
    elif placement_new.iloc[i]["Place"] <= 100:
        adjustment.at[i,"Place"] = "<=100"
    else:
        adjustment.at[i,"Place"] = ">100"

placement_new["Place"] = adjustment["Place"]
placement_new = placement_new.replace("DNF", np.nan)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
#print(placement_new)

C:\Users\gsant\AppData\Local\Temp\ipykernel_64452\751231791.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tied["Place"] = tied["Place"].str[:-2]
C:\Users\gsant\AppData\Local\Temp\ipykernel_64452\751231791.py:29: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '<=75' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  adjustment.at[i,"Place"] = "<=75"


In [301]:
# Clean cost data
cost_new = pd.DataFrame.copy(placement_new)
cost_new["Adjusted Cost"] = (cost_new["Adjusted Cost"].str.replace('[$, ]', '', regex=True).astype(float))
clean_data = pd.DataFrame.copy(cost_new)

In [302]:
# Ready acceleration for predictions
acceleration_prediction_data = clean_data.reset_index()
for i in range(len(acceleration_prediction_data)):
    if acceleration_prediction_data.iloc[i]["Acceleration Score"] > 90:
        acceleration_prediction_data.at[i,"Acceleration Score"] = ">90"
    elif acceleration_prediction_data.iloc[i]["Acceleration Score"] >= 80:
        acceleration_prediction_data.at[i,"Acceleration Score"] = ">=80"
    elif acceleration_prediction_data.iloc[i]["Acceleration Score"] >= 70:
        acceleration_prediction_data.at[i,"Acceleration Score"] = ">=70"
    elif acceleration_prediction_data.iloc[i]["Acceleration Score"] >= 60:
        acceleration_prediction_data.at[i,"Acceleration Score"] = ">=60"
    elif acceleration_prediction_data.iloc[i]["Acceleration Score"] >= 50:
        acceleration_prediction_data.at[i,"Acceleration Score"] = ">=50"
    elif not pd.isna(acceleration_prediction_data.iloc[i]["Acceleration Score"]):
        acceleration_prediction_data.at[i,"Acceleration Score"] = "<50"

clean_data["Acceleration Score"] = acceleration_prediction_data["Acceleration Score"]
print(clean_data["Acceleration Score"].value_counts())

Acceleration Score
>=70    16
>=80    15
>=60    13
>=50    13
<50     11
>90      8
Name: count, dtype: int64


C:\Users\gsant\AppData\Local\Temp\ipykernel_64452\3852708824.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '>=60' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  acceleration_prediction_data.at[i,"Acceleration Score"] = ">=60"


In [303]:
# # Ready acceleration for predictions
# acceleration_prediction_data = clean_data.reset_index()
# for i in range(len(acceleration_prediction_data)):
#     if acceleration_prediction_data.iloc[i]["Acceleration Score"] > 80:
#         acceleration_prediction_data.at[i,"Acceleration Score"] = ">80"
#     elif acceleration_prediction_data.iloc[i]["Acceleration Score"] >= 60:
#         acceleration_prediction_data.at[i,"Acceleration Score"] = ">=60"
#     elif not pd.isna(acceleration_prediction_data.iloc[i]["Acceleration Score"]):
#         acceleration_prediction_data.at[i,"Acceleration Score"] = "<60"

# clean_data["Acceleration Score"] = acceleration_prediction_data["Acceleration Score"]
# print(clean_data["Acceleration Score"].value_counts())

In [304]:
# Ready skidpad for predictions
skidpad_prediction_data = clean_data.reset_index()
for i in range(len(skidpad_prediction_data)):
    if skidpad_prediction_data.iloc[i]["Skid Pad Score"] >= 60:
        skidpad_prediction_data.at[i,"Skid Pad Score"] = ">=60"
    elif skidpad_prediction_data.iloc[i]["Skid Pad Score"] >= 50:
        skidpad_prediction_data.at[i,"Skid Pad Score"] = ">=50"
    elif skidpad_prediction_data.iloc[i]["Skid Pad Score"] >= 40:
        skidpad_prediction_data.at[i,"Skid Pad Score"] = ">=40"
    elif skidpad_prediction_data.iloc[i]["Skid Pad Score"] >= 30:
        skidpad_prediction_data.at[i,"Skid Pad Score"] = ">=30"
    elif skidpad_prediction_data.iloc[i]["Skid Pad Score"] >= 15:
        skidpad_prediction_data.at[i,"Skid Pad Score"] = ">=15"
    elif not pd.isna(skidpad_prediction_data.iloc[i]["Skid Pad Score"]):
        skidpad_prediction_data.at[i,"Skid Pad Score"] = "<15"

clean_data["Skid Pad Score"] = skidpad_prediction_data["Skid Pad Score"]
print(clean_data["Skid Pad Score"].value_counts())

Skid Pad Score
>=50    19
>=40    16
>=30    16
>=15    14
>=60     9
<15      4
Name: count, dtype: int64


C:\Users\gsant\AppData\Local\Temp\ipykernel_64452\1906675081.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '>=15' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  skidpad_prediction_data.at[i,"Skid Pad Score"] = ">=15"


In [305]:
# Ready Autocross for predictions
autocross_prediction_data = clean_data.reset_index()
for i in range(len(autocross_prediction_data)):
    if autocross_prediction_data.iloc[i]["Autocross Score"] > 100:
        autocross_prediction_data.at[i,"Autocross Score"] = ">100"
    elif autocross_prediction_data.iloc[i]["Autocross Score"] >= 90:
        autocross_prediction_data.at[i,"Autocross Score"] = ">=90"
    elif autocross_prediction_data.iloc[i]["Autocross Score"] >= 80:
        autocross_prediction_data.at[i,"Autocross Score"] = ">=80"
    elif autocross_prediction_data.iloc[i]["Autocross Score"] >= 70:
        autocross_prediction_data.at[i,"Autocross Score"] = ">=70"
    elif autocross_prediction_data.iloc[i]["Autocross Score"] >= 60:
        autocross_prediction_data.at[i,"Autocross Score"] = ">=60"
    elif autocross_prediction_data.iloc[i]["Autocross Score"] >= 50:
        autocross_prediction_data.at[i,"Autocross Score"] = ">=50"
    elif not pd.isna(autocross_prediction_data.iloc[i]["Autocross Score"]):
        autocross_prediction_data.at[i,"Autocross Score"] = "<50"

clean_data["Autocross Score"] = autocross_prediction_data["Autocross Score"]
print(clean_data["Autocross Score"].value_counts())

Autocross Score
>=90    16
<50     13
>=70    12
>=60    11
>100    10
>=80     9
>=50     8
Name: count, dtype: int64


C:\Users\gsant\AppData\Local\Temp\ipykernel_64452\3321792573.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '>=90' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  autocross_prediction_data.at[i,"Autocross Score"] = ">=90"


In [306]:
# Ready Endurance for predictions
endurance_prediction_data = clean_data.reset_index()
for i in range(len(endurance_prediction_data)):
    if endurance_prediction_data.iloc[i]["Endurance Score"] > 250:
        endurance_prediction_data.at[i,"Endurance Score"] = ">250"
    elif endurance_prediction_data.iloc[i]["Endurance Score"] >= 200:
        endurance_prediction_data.at[i,"Endurance Score"] = ">=200"
    elif endurance_prediction_data.iloc[i]["Endurance Score"] >= 175:
        endurance_prediction_data.at[i,"Endurance Score"] = ">=175"
    elif endurance_prediction_data.iloc[i]["Endurance Score"] >= 150:
        endurance_prediction_data.at[i,"Endurance Score"] = ">=150"
    elif endurance_prediction_data.iloc[i]["Endurance Score"] >= 125:
        endurance_prediction_data.at[i,"Endurance Score"] = ">=125"
    elif endurance_prediction_data.iloc[i]["Endurance Score"] >= 100:
        endurance_prediction_data.at[i,"Endurance Score"] = ">=100"
    elif endurance_prediction_data.iloc[i]["Endurance Score"] >= 50:
        endurance_prediction_data.at[i,"Endurance Score"] = ">=50"
    elif endurance_prediction_data.iloc[i]["Endurance Score"] >= 25:
        endurance_prediction_data.at[i,"Endurance Score"] = ">=25"
    elif endurance_prediction_data.iloc[i]["Endurance Score"] >= 10:
        endurance_prediction_data.at[i,"Endurance Score"] = ">=10"
    elif not pd.isna(endurance_prediction_data.iloc[i]["Endurance Score"]):
        endurance_prediction_data.at[i,"Endurance Score"] = "<25"

clean_data["Endurance Score"] = endurance_prediction_data["Endurance Score"]
print(clean_data["Endurance Score"].value_counts())
#print(clean_data)


Endurance Score
>=10     20
>=150    10
<25       9
>=100     8
>=200     6
>=175     5
>=125     5
>=25      5
>250      5
>=50      5
Name: count, dtype: int64


C:\Users\gsant\AppData\Local\Temp\ipykernel_64452\963217871.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '>=200' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  endurance_prediction_data.at[i,"Endurance Score"] = ">=200"


In [307]:
filepath = Path("fsae_data/clean_data/2025_clean.csv")
clean_data.to_csv(filepath, index= False)

In [308]:
# data load
base_data = pd.read_csv("./fsae_data/2024/fsae_2024_overall_results_clean.csv")

cost_data = pd.read_csv("./fsae_data/2024/fsae_2024_cost_event_results_clean.csv")
# cost_data = cost_data.drop(columns=["Place", "Car Num", "Penalty", "Score", "Penalty"])
cost_data = cost_data[["Team", "Adjusted Cost"]]
merged_data = pd.merge(base_data, cost_data, on = 'Team', how = "outer")

efficiency_data = pd.read_csv("./fsae_data/2024/fsae_2024_efficiency_results_clean.csv")
#efficiency_data = efficiency_data.drop(columns = ["Place", "Car Num", "Fuel Efficiency Score"])
efficiency_data = efficiency_data[["Team", "Average Adjusted Laptime", "Fuel Efficiency Factor"]]
merged_data = pd.merge(merged_data, efficiency_data, on= 'Team', how = 'outer')

presentation_data = pd.read_csv("./fsae_data/2024/fsae_2024_presentation_results.csv")
presentation_data = presentation_data.drop(columns = ["Place", "Car Num", "Score", "Penalty", "Raw Score", "Status"])
merged_data = pd.merge(merged_data, presentation_data, on= 'Team', how = 'outer')

accel_data = pd.read_csv("./fsae_data/2024/fsae_2024_acceleration_results_clean.csv", index_col=False)
accel_data = accel_data[["Team", "Best Time"]]
accel_data = accel_data.rename(columns = {"Final Best Time": "Best Acceleration Time"})
#accel_data = accel_data.drop(columns = ["Place", "Car Num", "Run 1 Time", "Run 1 Adj Time", "Run 2 Time", "Run 2 Adj Time", "Run 3 Time", "Run 3 Adj Time", "Run 4 Time", "Run 4 Adj Time"])
merged_data = pd.merge(merged_data, accel_data, on= 'Team', how = 'outer')

autocross_data = pd.read_csv("./fsae_data/2024/fsae_2024_autocross_results_clean.csv", index_col=False)
autocross_data = autocross_data[["Team", "Best Time"]]
autocross_data = autocross_data.rename(columns = {"Best Time": "Best Autocross Time"})
#autocross_data = autocross_data.drop(columns = ["Place", "Car Num", "Penalty","R1 Time","R1 Cones","R1 Off Course","R1 Adj Time","R2 Time","R2 Cones","R2 Off Course","R2 Adj Time","R3 Time","R3 Cones","R3 Off Course","R3 Adj Time","R4 Time","R4 Cones","R4 Off Course","R4 Adj Time"])
merged_data = pd.merge(merged_data, autocross_data, on= 'Team', how = 'outer')

endurance_data = pd.read_csv("./fsae_data/2024/fsae_2024_endurance_results_clean.csv", index_col=False)
#endurance_data = endurance_data.drop(columns = ["Place", "Car Num", "Cones", "Endurance Score"])
endurance_data = endurance_data[["Team", "Time", "Other Penalty", "Adjusted Time", "Time Score", "Laps", "Laps Score"]]
endurance_data = endurance_data.rename(columns = {"Time": "Endurance Time", "Other Penalty": "Endurance Penalty", "Adjusted Time": "Adjusted Endurance Time", "Time Score": "Endurance Time Score", "Laps": "Endurance Laps", "Laps Score": "Endurance Laps Score"})
merged_data = pd.merge(merged_data, endurance_data, on= 'Team', how = 'outer')

skidpad_data = pd.read_csv("./fsae_data/2024/fsae_2024_skidpad_results_clean.csv", index_col=False)
skidpad_data = skidpad_data[["Team", "Best Time"]]
skidpad_data = skidpad_data.rename(columns = {"Best Time": "Best Skidpad Time"})
merged_data = pd.merge(merged_data, skidpad_data, on= 'Team', how = 'outer')


team_data = pd.read_csv("./fsae_data/2024/fsae_2024_team_information_clean.csv", index_col=False)
team_data = team_data[["Team", "Engine Cylinders", "Engine Displacement (cc)", "Weight (kg)"]]
merged_data = pd.merge(merged_data, team_data, on= 'Team', how = 'outer')

merged_data["Team"] = merged_data["Team"] + "_2024"
#print(merged_data)

In [309]:
# Clean place data
placement_new = pd.DataFrame.copy(merged_data)
placement_new = placement_new.dropna(subset=["Place"])
unranked = merged_data[merged_data["Place"] == "Unranked"]
placement_new = placement_new[placement_new.Place != "Unranked"]

placement_new["Place"] = placement_new["Place"].astype(int)
placement_new = placement_new.reset_index(drop = True)
adjustment = pd.DataFrame.copy(placement_new)
for i in range(len(placement_new)):
    if placement_new.iloc[i]["Place"] == 1:
        adjustment.at[i,"Place"] = "1"
    elif placement_new.iloc[i]["Place"] <= 5:
        adjustment.at[i,"Place"] = "<=5"
    elif placement_new.iloc[i]["Place"] <= 10:
        adjustment.at[i,"Place"] = "<=10"
    elif placement_new.iloc[i]["Place"] <= 25:
        adjustment.at[i,"Place"] = "<=25"
    elif placement_new.iloc[i]["Place"] <= 50:
        adjustment.at[i,"Place"] = "<=50"
    elif placement_new.iloc[i]["Place"] <= 75:
        adjustment.at[i,"Place"] = "<=75"
    elif placement_new.iloc[i]["Place"] <= 100:
        adjustment.at[i,"Place"] = "<=100"
    else:
        adjustment.at[i,"Place"] = ">100"

placement_new["Place"] = adjustment["Place"]
clean_data = placement_new.replace("DNF", np.nan)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
#print(clean_data)

C:\Users\gsant\AppData\Local\Temp\ipykernel_64452\1163459188.py:18: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '<=25' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  adjustment.at[i,"Place"] = "<=25"


In [310]:
# Ready acceleration for predictions
acceleration_prediction_data = clean_data.reset_index()
for i in range(len(acceleration_prediction_data)):
    if acceleration_prediction_data.iloc[i]["Acceleration Score"] > 90:
        acceleration_prediction_data.at[i,"Acceleration Score"] = ">90"
    elif acceleration_prediction_data.iloc[i]["Acceleration Score"] >= 80:
        acceleration_prediction_data.at[i,"Acceleration Score"] = ">=80"
    elif acceleration_prediction_data.iloc[i]["Acceleration Score"] >= 70:
        acceleration_prediction_data.at[i,"Acceleration Score"] = ">=70"
    elif acceleration_prediction_data.iloc[i]["Acceleration Score"] >= 60:
        acceleration_prediction_data.at[i,"Acceleration Score"] = ">=60"
    elif acceleration_prediction_data.iloc[i]["Acceleration Score"] >= 50:
        acceleration_prediction_data.at[i,"Acceleration Score"] = ">=50"
    elif not pd.isna(acceleration_prediction_data.iloc[i]["Acceleration Score"]):
        acceleration_prediction_data.at[i,"Acceleration Score"] = "<50"

clean_data["Acceleration Score"] = acceleration_prediction_data["Acceleration Score"]
print(clean_data["Acceleration Score"].value_counts())

Acceleration Score
>=70    19
<50     13
>=60    12
>=50    10
>=80     9
>90      8
Name: count, dtype: int64


C:\Users\gsant\AppData\Local\Temp\ipykernel_64452\3852708824.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '>=70' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  acceleration_prediction_data.at[i,"Acceleration Score"] = ">=70"


In [311]:
# Ready skidpad for predictions
skidpad_prediction_data = clean_data.reset_index()
for i in range(len(skidpad_prediction_data)):
    if skidpad_prediction_data.iloc[i]["Skid Pad Score"] >= 60:
        skidpad_prediction_data.at[i,"Skid Pad Score"] = ">=60"
    elif skidpad_prediction_data.iloc[i]["Skid Pad Score"] >= 50:
        skidpad_prediction_data.at[i,"Skid Pad Score"] = ">=50"
    elif skidpad_prediction_data.iloc[i]["Skid Pad Score"] >= 40:
        skidpad_prediction_data.at[i,"Skid Pad Score"] = ">=40"
    elif skidpad_prediction_data.iloc[i]["Skid Pad Score"] >= 30:
        skidpad_prediction_data.at[i,"Skid Pad Score"] = ">=30"
    elif skidpad_prediction_data.iloc[i]["Skid Pad Score"] >= 15:
        skidpad_prediction_data.at[i,"Skid Pad Score"] = ">=15"
    elif not pd.isna(skidpad_prediction_data.iloc[i]["Skid Pad Score"]):
        skidpad_prediction_data.at[i,"Skid Pad Score"] = "<15"

clean_data["Skid Pad Score"] = skidpad_prediction_data["Skid Pad Score"]
print(clean_data["Skid Pad Score"].value_counts())

Skid Pad Score
>=50    17
>=40    17
>=60    16
>=30    12
<15      9
>=15     6
Name: count, dtype: int64


C:\Users\gsant\AppData\Local\Temp\ipykernel_64452\1906675081.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '>=50' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  skidpad_prediction_data.at[i,"Skid Pad Score"] = ">=50"


In [312]:
# Ready Autocross for predictions
autocross_prediction_data = clean_data.reset_index()
for i in range(len(autocross_prediction_data)):
    if autocross_prediction_data.iloc[i]["Autocross Score"] > 100:
        autocross_prediction_data.at[i,"Autocross Score"] = ">100"
    elif autocross_prediction_data.iloc[i]["Autocross Score"] >= 90:
        autocross_prediction_data.at[i,"Autocross Score"] = ">=90"
    elif autocross_prediction_data.iloc[i]["Autocross Score"] >= 80:
        autocross_prediction_data.at[i,"Autocross Score"] = ">=80"
    elif autocross_prediction_data.iloc[i]["Autocross Score"] >= 70:
        autocross_prediction_data.at[i,"Autocross Score"] = ">=70"
    elif autocross_prediction_data.iloc[i]["Autocross Score"] >= 60:
        autocross_prediction_data.at[i,"Autocross Score"] = ">=60"
    elif autocross_prediction_data.iloc[i]["Autocross Score"] >= 50:
        autocross_prediction_data.at[i,"Autocross Score"] = ">=50"
    elif not pd.isna(autocross_prediction_data.iloc[i]["Autocross Score"]):
        autocross_prediction_data.at[i,"Autocross Score"] = "<50"

clean_data["Autocross Score"] = autocross_prediction_data["Autocross Score"]
print(clean_data["Autocross Score"].value_counts())

Autocross Score
<50     32
>=60    15
>=70    10
>=50    10
>=90     7
>=80     4
>100     2
Name: count, dtype: int64


C:\Users\gsant\AppData\Local\Temp\ipykernel_64452\3321792573.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '>=70' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  autocross_prediction_data.at[i,"Autocross Score"] = ">=70"


In [313]:
# Ready Endurance for predictions
endurance_prediction_data = clean_data.reset_index()
for i in range(len(endurance_prediction_data)):
    if endurance_prediction_data.iloc[i]["Endurance Score"] > 250:
        endurance_prediction_data.at[i,"Endurance Score"] = ">250"
    elif endurance_prediction_data.iloc[i]["Endurance Score"] >= 200:
        endurance_prediction_data.at[i,"Endurance Score"] = ">=200"
    elif endurance_prediction_data.iloc[i]["Endurance Score"] >= 175:
        endurance_prediction_data.at[i,"Endurance Score"] = ">=175"
    elif endurance_prediction_data.iloc[i]["Endurance Score"] >= 150:
        endurance_prediction_data.at[i,"Endurance Score"] = ">=150"
    elif endurance_prediction_data.iloc[i]["Endurance Score"] >= 125:
        endurance_prediction_data.at[i,"Endurance Score"] = ">=125"
    elif endurance_prediction_data.iloc[i]["Endurance Score"] >= 100:
        endurance_prediction_data.at[i,"Endurance Score"] = ">=100"
    elif endurance_prediction_data.iloc[i]["Endurance Score"] >= 50:
        endurance_prediction_data.at[i,"Endurance Score"] = ">=50"
    elif endurance_prediction_data.iloc[i]["Endurance Score"] >= 25:
        endurance_prediction_data.at[i,"Endurance Score"] = ">=25"
    elif endurance_prediction_data.iloc[i]["Endurance Score"] >= 10:
        endurance_prediction_data.at[i,"Endurance Score"] = ">=10"
    elif not pd.isna(endurance_prediction_data.iloc[i]["Endurance Score"]):
        endurance_prediction_data.at[i,"Endurance Score"] = "<25"

clean_data["Endurance Score"] = endurance_prediction_data["Endurance Score"]
print(clean_data["Endurance Score"].value_counts())
print(clean_data)


C:\Users\gsant\AppData\Local\Temp\ipykernel_64452\788848518.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '>=125' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  endurance_prediction_data.at[i,"Endurance Score"] = ">=125"


Endurance Score
>=10     15
>=25     12
<25      10
>=125     9
>=50      6
>=200     5
>=100     5
>=150     5
>=175     4
>250      3
Name: count, dtype: int64
     Place  Car Num                                           Team  Penalty  \
0     <=25     68.0        AGH Univ of Science and Technology_2024      NaN   
1     <=50    126.0                Arizona State Univ - Tempe_2024      NaN   
2    <=100     69.0                              Bradley Univ_2024      NaN   
3     <=50     24.0                                Brown Univ_2024      NaN   
4     <=50    119.0             California Baptist University_2024      NaN   
5     <=10      6.0     California Polytechnic State Univ-SLO_2024      NaN   
6     <=50     25.0       California State Poly Univ - Pomona_2024      NaN   
7    <=100     52.0             California State Univ - Chico_2024      NaN   
8     <=75     80.0         California State Univ - Fullerton_2024      NaN   
9     <=25     45.0        California State Univ

In [314]:
clean_data.replace("DNA", "", inplace=True)
filepath = Path("fsae_data/clean_data/2024_clean.csv")
clean_data.to_csv(filepath, index= False)

In [315]:
data_2025 = pd.read_csv("./fsae_data/clean_data/2025_clean.csv")
data_2024 = pd.read_csv("./fsae_data/clean_data/2024_clean.csv")
ultra_data = pd.concat([data_2025, data_2024], axis = 0, ignore_index=True)
filepath = Path("fsae_data/clean_data/ultra_clean.csv")
ultra_data.to_csv(filepath, index= False)
